In [1]:
import pandas as pd
import os

In [16]:
base_dir = "C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/강우량"

# 3개년 여름철 데이터 통합 (2023, 2024, 2025년 6, 7, 8월)
years = [2023, 2024, 2025]
months = ['06', '07', '08']

df_list = []

for y in years:
    for m in months:
        file_name = f'서울시_강우량_정보_{y}년{m}월.csv'
        file_path = f"{base_dir}/{file_name}"
        
        # 파일이 존재하는 경우에만 읽어오기
        if os.path.exists(file_path):
            try:
                temp_df = pd.read_csv(file_path, encoding='cp949')
            except UnicodeDecodeError:
                temp_df = pd.read_csv(file_path, encoding='utf-8')
            
            df_list.append(temp_df)
        else:
            print(f"경고: {file_name} 파일을 찾을 수 없습니다.")

In [17]:
# 모든 데이터프레임을 하나로 병합
rain_df = pd.concat(df_list, ignore_index=True)
print(f"총 데이터 건수: {len(rain_df):,}건")

총 데이터 건수: 1,883,955건


In [18]:
# 2. 시계열 데이터 전처리
# '자료수집 시각'에 분 단위와 초 단위 형식이 섞여 있으므로 format='mixed' 적용
# 변환할 수 없는 이상한 문자열은 강제로 결측치(NaT)로 만든 후 제거
rain_df['자료수집 시각'] = pd.to_datetime(rain_df['자료수집 시각'], format='mixed', errors='coerce')
rain_df = rain_df.dropna(subset=['자료수집 시각'])

# 결측치나 이상치(예: 강우량이 음수인 경우 등) 제거
rain_df = rain_df[rain_df['10분우량'] >= 0]

In [19]:
# 3. 강우량계별 집중호우 지표 산출 (일 최대, 시간 최대 강수량)
# 시간 연산을 위해 인덱스를 시간으로 설정
rain_df.set_index('자료수집 시각', inplace=True)

In [20]:
# 3-1. 관측소별 시간당 강수량 (1H 단위 합계)
hourly_rain = rain_df.groupby(['구청명', '강우량계명'])['10분우량'].resample('1H').sum().reset_index()

# 3-2. 관측소별 일일 강수량 (1D 단위 합계)
daily_rain = rain_df.groupby(['구청명', '강우량계명'])['10분우량'].resample('1D').sum().reset_index()

C:\Users\hyeon\AppData\Local\Temp\ipykernel_13224\4016918390.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_rain = rain_df.groupby(['구청명', '강우량계명'])['10분우량'].resample('1H').sum().reset_index()


In [21]:
# 관측소별 전체 기간 중 '최대 시간당 강수량'과 '최대 일일 강수량' 추출
max_hourly_per_gauge = hourly_rain.groupby(['구청명', '강우량계명'])['10분우량'].max().reset_index(name='최대_시우량')
max_daily_per_gauge = daily_rain.groupby(['구청명', '강우량계명'])['10분우량'].max().reset_index(name='최대_일강수량')

In [22]:
# 두 지표 병합
gauge_features = pd.merge(max_hourly_per_gauge, max_daily_per_gauge, on=['구청명', '강우량계명'])

In [23]:
# 4. 자치구(구청명)별 노출 지표 집계
# 한 자치구 내에 강우량계가 여러 개일 수 있으므로, 자치구 내 최댓값을 사용
gu_exposure = gauge_features.groupby('구청명')[['최대_시우량', '최대_일강수량']].max().reset_index()

# 열 이름 정리 ('구청명' -> '자치구')
gu_exposure.rename(columns={'구청명': '자치구'}, inplace=True)

In [12]:
# 5. 노출 지표 정규화 
# 최대 시우량 정규화
h_min = gu_exposure['최대_시우량'].min()
h_max = gu_exposure['최대_시우량'].max()
gu_exposure['E_최대시우량_정규화'] = (gu_exposure['최대_시우량'] - h_min) / (h_max - h_min)

In [13]:
# 최대 일강수량 정규화
d_min = gu_exposure['최대_일강수량'].min()
d_max = gu_exposure['최대_일강수량'].max()
gu_exposure['E_최대일강수량_정규화'] = (gu_exposure['최대_일강수량'] - d_min) / (d_max - d_min)

In [14]:
# 최종 노출 지수 (E) 산출: 두 지표의 평균 사용
gu_exposure['E_최종지수'] = (gu_exposure['E_최대시우량_정규화'] + gu_exposure['E_최대일강수량_정규화']) / 2

In [15]:
# 6. 결과 확인 (위험도가 높은 자치구 순으로 정렬)
gu_exposure_sorted = gu_exposure.sort_values(by='E_최종지수', ascending=False).reset_index(drop=True)
print("\n[자치구별 기후 노출(E) 지수 산출 결과]")
display(gu_exposure_sorted.head())


[자치구별 기후 노출(E) 지수 산출 결과]


,자치구,최대_시우량,최대_일강수량,E_최대시우량_정규화,E_최대일강수량_정규화,E_최종지수
0,구로구,453.5,464.5,1.000000,1.000000,1.000000
1,송파구,295.0,295.0,0.624852,0.504386,0.564619
2,은평구,104.0,218.0,0.172781,0.279240,0.226010
3,노원구,60.0,238.5,0.068639,0.339181,0.203910
4,도봉구,57.5,232.0,0.062722,0.320175,0.191449


### 1. 집중호우 지표 산출
- 10분 단위 우량 -> 홍수에 직접 타격을 주는 극한 강수 지표로 리샘플링
    - 일 최대 강수량 : 24시간동안 비가 가장 많이 내린 날의 강수량
    - 시우량 : 1시간동안 내린 비의 양
    - 총 강수량 : 해당 연도 여름동안 내린 전체 누적 강수량

### 2. 위치 기반 매핑 및 자치구별 집계
- 반지하비율(S1)과 맞춰 자치구별로 정리
    - 데이터에 포함되어 있는 구청명 기준으로 groupby
    - 해당 자치구 내 측정소들의 평균 또는 최댓값 (동일 구 안에 관측소 여러개)

### 3. 취약성 평가 지수
- 일 최대 강수량, 시간 최대 강수량을 정규화 후 평균 : 최종 E
    - 홍수 취약성 공식에서 노출 지표(Exposure, E)

In [25]:
# 불투수면적 데이터 로드 및 병합
imperv = pd.read_excel('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/토지/불투수면적_현황자료_2025.xlsx')

In [26]:
# 필요 칼럼 추출
imperv = imperv[['자치구', '불투수면적 비율(%)']]

In [27]:
# 기존 강우량 노출 지표(gu_exposure)와 불투수면적 데이터 병합
runoff_df = pd.merge(gu_exposure, imperv, on='자치구', how='left')

In [ ]:
# 유효 강수량(표면유출량) 추정치 산출
# 땅에 흡수되지 못하고 하수관거로 쏟아지는 실제 물의 양
# 공식: 강수량 * (불투수면적 비율 / 100)
runoff_df['유효_최대시우량'] = runoff_df['최대_시우량'] * (runoff_df['불투수면적 비율(%)'] / 100)
runoff_df['유효_최대일강수량'] = runoff_df['최대_일강수량'] * (runoff_df['불투수면적 비율(%)'] / 100)

In [29]:
# 표면유출 위험지수 정규화
# 유효_최대시우량 정규화
eff_h_min = runoff_df['유효_최대시우량'].min()
eff_h_max = runoff_df['유효_최대시우량'].max()
runoff_df['유효_최대시우량_정규화'] = (runoff_df['유효_최대시우량'] - eff_h_min) / (eff_h_max - eff_h_min)

In [30]:
# 유효_최대일강수량 정규화
eff_d_min = runoff_df['유효_최대일강수량'].min()
eff_d_max = runoff_df['유효_최대일강수량'].max()
runoff_df['유효_최대일강수량_정규화'] = (runoff_df['유효_최대일강수량'] - eff_d_min) / (eff_d_max - eff_d_min)

In [31]:
# 최종 결합 지수 산출: 두 유효 우량 지표의 평균
runoff_df['표면유출_위험지수'] = (runoff_df['유효_최대시우량_정규화'] + runoff_df['유효_최대일강수량_정규화']) / 2

In [33]:
# 결과 확인 (표면유출 위험이 가장 큰 자치구 순 정렬)
runoff_sorted = runoff_df.sort_values(by='표면유출_위험지수', ascending=False).reset_index(drop=True)

print("\n[자치구별 강수 노출 * 불투수면적 결합 지수 결과]")
display(runoff_sorted[['자치구', '최대_시우량', '불투수면적 비율(%)', '유효_최대시우량', '표면유출_위험지수']].head(10))


[자치구별 강수 노출 * 불투수면적 결합 지수 결과]


,자치구,최대_시우량,불투수면적 비율(%),유효_최대시우량,표면유출_위험지수
0,구로구,453.5,61.49,278.85715,1.000000
1,송파구,295.0,57.87,170.71650,0.553733
2,동작구,88.0,61.61,54.21680,0.158287
3,영등포구,95.5,59.49,56.81295,0.143272
4,양천구,70.5,67.12,47.31960,0.139798
5,중구,31.0,75.17,23.30270,0.132573
6,동대문구,41.0,70.90,29.06900,0.132015
7,은평구,104.0,39.07,40.63280,0.126891
8,금천구,65.0,60.37,39.24050,0.122867
9,도봉구,57.5,40.05,23.02875,0.109463


### 1. 표면유출량(유효 강수량) 추정
- 강수량과 불투수면적을 결합해 실제 도심 표면으로 흘러 넘치는 빗물의 양 산출
    - 유효 최대 시우량
        - 시간당 최대 강수량 * 불투수면적 비율
        - 단기간에 땅에 스며들지 못하는 빗물의 양
    - 유효 최대 일강수량
        - 일 최대 강수량 * 불투수면적 비율
        - 하루동안 침수 취약 구역에 지속적으로 누적되는 빗물의 양

### 2. 위치 기반 데이터 결합
- 기후 지표와 토지피복 지표를 동일한 기준으로 병합
    - 자치구 기준으로 병합

### 3. 표면 유출 위험지수 산출
- 유효 시우량과 유효 일강수량을 각각 정규화 후 평균
    - 최종 표면유출 위험지수